# 01 — Exploration des données brutes (Raw)
## Analyse du fichier `wti_history_init.parquet`

Ce notebook charge le fichier Parquet produit par le script d'ingestion Yahoo Finance et effectue une **analyse complète** :

1. **Chargement** et aperçu du DataFrame
2. **Statistiques descriptives** (min, max, moyenne, écart-type…)
3. **Vérification de la qualité** (valeurs manquantes, doublons)
4. **Analyse temporelle** (volume de données par jour/heure)
5. **Visualisations** (prix de clôture, chandeliers, volumes, distribution)

In [1]:
# ═══════════════════════════════════════════════
# 📦 Imports
# ═══════════════════════════════════════════════
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Style des graphiques
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

# Chemin du fichier Parquet
PARQUET_PATH = os.path.join("..", "raw", "yahoofinance", "history", "wti_history_init.parquet")
print(f"Fichier ciblé : {os.path.abspath(PARQUET_PATH)}")

Fichier ciblé : /home/omarf/Projet_DataLake/geopolitics-oil-data-project/raw/yahoofinance/history/wti_history_init.parquet


## 1. Chargement et aperçu du DataFrame

In [2]:
# ═══════════════════════════════════════════════
# 1. Chargement du fichier Parquet
# ═══════════════════════════════════════════════
df = pd.read_parquet(PARQUET_PATH, engine="pyarrow")

print(f"Shape : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(df.dtypes)
print(f"\n{'═' * 60}")
print("Aperçu des 5 premières lignes :")
df.head()

Shape : 2939 lignes × 6 colonnes

Types des colonnes :
Datetime    datetime64[ns, UTC]
Close                   float64
High                    float64
Low                     float64
Open                    float64
Volume                    int64
dtype: object

════════════════════════════════════════════════════════════
Aperçu des 5 premières lignes :


,Datetime,Close,High,Low,Open,Volume
0,2026-01-05 05:00:00+00:00,57.060001,57.060001,56.990002,57.020000,256
1,2026-01-05 05:15:00+00:00,57.060001,57.099998,57.040001,57.049999,375
2,2026-01-05 05:30:00+00:00,56.930000,57.080002,56.910000,57.070000,1057
3,2026-01-05 05:45:00+00:00,56.889999,56.930000,56.840000,56.919998,711
4,2026-01-05 06:00:00+00:00,56.980000,56.980000,56.869999,56.889999,730


In [3]:
# Dernières lignes
print("Aperçu des 5 dernières lignes :")
df.tail()

Aperçu des 5 dernières lignes :


,Datetime,Close,High,Low,Open,Volume
2934,2026-02-25 00:45:00+00:00,66.099998,66.209999,66.059998,66.190002,408
2935,2026-02-25 01:00:00+00:00,66.160004,66.300003,66.029999,66.089996,2009
2936,2026-02-25 01:15:00+00:00,66.120003,66.209999,66.110001,66.169998,1318
2937,2026-02-25 01:30:00+00:00,66.040001,66.120003,65.949997,66.120003,1093
2938,2026-02-25 01:45:00+00:00,66.070000,66.139999,66.040001,66.050003,989


## 2. Statistiques descriptives

In [4]:
# ═══════════════════════════════════════════════
# 2. Statistiques descriptives complètes
# ═══════════════════════════════════════════════
print("Statistiques descriptives :\n")
df.describe().round(2)

Statistiques descriptives :



,Close,High,Low,Open,Volume
count,2939.00,2939.00,2939.00,2939.00,2939.00
mean,61.78,61.88,61.68,61.78,3703.25
std,2.68,2.69,2.66,2.68,6772.08
min,55.87,55.97,55.76,55.87,25.00
25%,59.71,59.78,59.62,59.71,689.00
50%,62.18,62.25,62.05,62.17,2025.00
75%,64.02,64.13,63.91,64.02,4882.00
max,67.02,67.15,66.86,67.02,260246.00


In [5]:
# ═══════════════════════════════════════════════
# Informations sur la plage temporelle
# ═══════════════════════════════════════════════
print(f"Plage temporelle : {df['Datetime'].min()} → {df['Datetime'].max()}")
print(f"Durée totale     : {df['Datetime'].max() - df['Datetime'].min()}")
print(f"Nombre de jours uniques : {df['Datetime'].dt.date.nunique()}")

# Prix WTI — résumé rapide
print(f"\n{'═' * 60}")
print(f"Prix WTI (Close) :")
print(f"  Min      : {df['Close'].min():.2f} $")
print(f"  Max      : {df['Close'].max():.2f} $")
print(f"  Moyenne  : {df['Close'].mean():.2f} $")
print(f"  Médiane  : {df['Close'].median():.2f} $")
print(f"  Écart-type : {df['Close'].std():.2f} $")
print(f"  Amplitude  : {df['Close'].max() - df['Close'].min():.2f} $")

Plage temporelle : 2026-01-05 05:00:00+00:00 → 2026-02-25 01:45:00+00:00
Durée totale     : 50 days 20:45:00
Nombre de jours uniques : 42

════════════════════════════════════════════════════════════
Prix WTI (Close) :
  Min      : 55.87 $
  Max      : 67.02 $
  Moyenne  : 61.78 $
  Médiane  : 62.18 $
  Écart-type : 2.68 $
  Amplitude  : 11.15 $


## 3. Qualité des données (valeurs manquantes, doublons)

In [6]:
# ═══════════════════════════════════════════════
# 3. Qualité des données
# ═══════════════════════════════════════════════

# 3a. Valeurs manquantes par colonne
print("Valeurs manquantes (NaN) par colonne :")
missing = df.isnull().sum()
print(missing.to_string())
print(f"\nTotal NaN : {missing.sum()} / {df.shape[0] * df.shape[1]} cellules ({missing.sum() / (df.shape[0] * df.shape[1]) * 100:.2f}%)")

# 3b. Lignes entièrement dupliquées
doublons = df.duplicated().sum()
print(f"\n{'═' * 60}")
print(f"Lignes dupliquées : {doublons}")

# 3c. Doublons sur Datetime uniquement (même timestamp)
doublons_datetime = df.duplicated(subset=["Datetime"]).sum()
print(f"Timestamps en double : {doublons_datetime}")

# 3d. Vérification de la continuité temporelle
print(f"\n{'═' * 60}")
print("Continuité temporelle :")
df_sorted = df.sort_values("Datetime")
time_diffs = df_sorted["Datetime"].diff().dropna()
print(f"  Intervalle le plus fréquent : {time_diffs.mode().iloc[0]}")
print(f"  Intervalle min              : {time_diffs.min()}")
print(f"  Intervalle max              : {time_diffs.max()} (gaps = fermeture marché)")

Valeurs manquantes (NaN) par colonne :
Datetime    0
Close       0
High        0
Low         0
Open        0
Volume      0

Total NaN : 0 / 17634 cellules (0.00%)

════════════════════════════════════════════════════════════
Lignes dupliquées : 0
Timestamps en double : 0

════════════════════════════════════════════════════════════
Continuité temporelle :
  Intervalle le plus fréquent : 0 days 00:15:00
  Intervalle min              : 0 days 00:15:00
  Intervalle max              : 4 days 01:30:00 (gaps = fermeture marché)


## 4. Analyse temporelle (volume de données par jour / heure)

In [ ]:
# ═══════════════════════════════════════════════
# 4. Analyse temporelle — Nombre de bougies par jour
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 4a. Nombre de lignes par jour
daily_counts = df.set_index("Datetime").resample("D").size()
daily_counts = daily_counts[daily_counts > 0]  # Exclure les jours sans données

axes[0].bar(daily_counts.index, daily_counts.values, color="steelblue", alpha=0.8)
axes[0].set_title("Nombre de bougies (15min) par jour", fontweight="bold")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Nombre de bougies")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
axes[0].tick_params(axis="x", rotation=45)

# 4b. Distribution par heure de la journée (UTC)
df["Heure"] = df["Datetime"].dt.hour
hourly_counts = df.groupby("Heure").size()

axes[1].bar(hourly_counts.index, hourly_counts.values, color="coral", alpha=0.8)
axes[1].set_title("Distribution des données par heure (UTC)", fontweight="bold")
axes[1].set_xlabel("Heure (UTC)")
axes[1].set_ylabel("Nombre de bougies")
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

print(f"Jours de trading dans la période : {len(daily_counts)}")
print(f"Moyenne de bougies par jour : {daily_counts.mean():.0f}")

## 5. Visualisations

### 5a. Évolution du prix de clôture (Close)

In [ ]:
# ═══════════════════════════════════════════════
# 5a. Évolution du prix de clôture WTI
# ═══════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df["Datetime"], df["Close"], linewidth=0.6, color="royalblue", alpha=0.9)
ax.fill_between(df["Datetime"], df["Close"], alpha=0.15, color="royalblue")

# Moyenne mobile 24h (96 bougies de 15min)
ma_24h = df["Close"].rolling(window=96, min_periods=1).mean()
ax.plot(df["Datetime"], ma_24h, linewidth=1.5, color="orange", label="Moyenne mobile 24h")

ax.set_title("Prix de clôture WTI — Intervalle 15 minutes", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Prix (USD)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 5b. Graphique en chandeliers (Candlestick) — Derniers 5 jours

In [ ]:
# ═══════════════════════════════════════════════
# 5b. Chandeliers japonais (derniers 10 jours)
# ═══════════════════════════════════════════════
from matplotlib.patches import FancyBboxPatch

# Filtrer les 5 derniers jours de trading
last_dates = sorted(df["Datetime"].dt.date.unique())[-10:]
df_last = df[df["Datetime"].dt.date.isin(last_dates)].copy()

fig, ax = plt.subplots(figsize=(16, 6))

width = 0.005  # Largeur des bougies
for idx, row in df_last.iterrows():
    color = "green" if row["Close"] >= row["Open"] else "red"
    # Mèche (High - Low)
    ax.plot([row["Datetime"], row["Datetime"]], [row["Low"], row["High"]],
            color=color, linewidth=0.6)
    # Corps (Open - Close)
    ax.bar(row["Datetime"], row["Close"] - row["Open"], bottom=row["Open"],
           width=width, color=color, alpha=0.8, edgecolor=color)

ax.set_title(f"Chandeliers WTI 15min — {last_dates[0]} → {last_dates[-1]}", fontsize=14, fontweight="bold")
ax.set_xlabel("Date / Heure (UTC)")
ax.set_ylabel("Prix (USD)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m %H:%M"))
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 5c. Zoom sur une journée — pause quotidienne de clôture (17h00–18h00 ET)

### 5c. Volume de trading

In [ ]:
# ═══════════════════════════════════════════════
# 5c. Volume de trading journalier
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Graphique du prix
axes[0].plot(df["Datetime"], df["Close"], linewidth=0.6, color="royalblue")
axes[0].set_title("Prix de clôture WTI (15min)", fontweight="bold")
axes[0].set_ylabel("Prix (USD)")

# Graphique du volume
axes[1].bar(df["Datetime"], df["Volume"], width=0.008, color="teal", alpha=0.7)
axes[1].set_title("Volume de trading", fontweight="bold")
axes[1].set_ylabel("Volume")
axes[1].set_xlabel("Date")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Volume total
print(f"Volume total sur la période : {df['Volume'].sum():,.0f}")
print(f"Volume moyen par bougie     : {df['Volume'].mean():,.0f}")

### 5d. Distribution des prix de clôture et rendements

In [ ]:
# ═══════════════════════════════════════════════
# 5d. Distribution des prix et rendements
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution du prix de clôture
axes[0].hist(df["Close"], bins=50, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].axvline(df["Close"].mean(), color="red", linestyle="--", linewidth=1.5, label=f"Moyenne: {df['Close'].mean():.2f}$")
axes[0].axvline(df["Close"].median(), color="orange", linestyle="--", linewidth=1.5, label=f"Médiane: {df['Close'].median():.2f}$")
axes[0].set_title("Distribution des prix de clôture", fontweight="bold")
axes[0].set_xlabel("Prix (USD)")
axes[0].set_ylabel("Fréquence")
axes[0].legend()

# Distribution des rendements (% change)
df["Rendement"] = df["Close"].pct_change() * 100  # En pourcentage
rendements = df["Rendement"].dropna()

axes[1].hist(rendements, bins=80, color="coral", alpha=0.8, edgecolor="white")
axes[1].axvline(0, color="black", linestyle="-", linewidth=1)
axes[1].set_title("Distribution des rendements (15min)", fontweight="bold")
axes[1].set_xlabel("Rendement (%)")
axes[1].set_ylabel("Fréquence")

plt.tight_layout()
plt.show()

print(f"Rendement moyen (15min)   : {rendements.mean():.4f}%")
print(f"Rendement médian          : {rendements.median():.4f}%")
print(f"Volatilité (écart-type)   : {rendements.std():.4f}%")
print(f"Rendement min             : {rendements.min():.4f}%")
print(f"Rendement max             : {rendements.max():.4f}%")

### 5e. Volatilité glissante (Rolling Std 24h)

In [ ]:
# ═══════════════════════════════════════════════
# 5e. Volatilité glissante sur 24h (96 bougies de 15min)
# ═══════════════════════════════════════════════
df_vol = df.copy().sort_values("Datetime")
rendements_vol = df_vol["Close"].pct_change() * 100

# Volatilité rolling : écart-type des rendements sur fenêtre de 96 bougies (= 24h)
rolling_vol = rendements_vol.rolling(window=96, min_periods=20).std()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Prix
axes[0].plot(df_vol["Datetime"], df_vol["Close"], linewidth=0.6, color="royalblue")
axes[0].set_title("Prix de clôture WTI (15min)", fontweight="bold")
axes[0].set_ylabel("Prix (USD)")

# Volatilité rolling
axes[1].plot(df_vol["Datetime"], rolling_vol, linewidth=1.0, color="crimson", alpha=0.85)
axes[1].fill_between(df_vol["Datetime"], rolling_vol, alpha=0.2, color="crimson")
axes[1].axhline(rolling_vol.mean(), color="black", linestyle="--", linewidth=1,
                label=f"Volatilité moyenne : {rolling_vol.mean():.3f}%")
axes[1].set_title("Volatilité glissante 24h (Rolling Std des rendements 15min)", fontweight="bold")
axes[1].set_ylabel("Volatilité (%)")
axes[1].set_xlabel("Date")
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"Volatilité moyenne 24h  : {rolling_vol.mean():.4f}%")
print(f"Volatilité max 24h      : {rolling_vol.max():.4f}%")
print(f"Volatilité min 24h      : {rolling_vol.min():.4f}%")

---

## 6. Conclusions

### Résumé de l'analyse du fichier `wti_history_init.parquet`

| Indicateur | Valeur |
|---|---|
| **Source** | Yahoo Finance — Ticker `CL=F` (WTI Crude Oil) |
| **Période couverte** | 05/01/2026 → 25/02/2026 |
| **Granularité** | 15 minutes |
| **Volume de données** | ~2 939 lignes × 6 colonnes |
| **Colonnes** | `Datetime`, `Open`, `High`, `Low`, `Close`, `Volume` |

### Points clés

- **Qualité des données** : 0 valeur manquante, 0 doublon — fichier propre et exploitable directement
- **Couverture temporelle** : Les gaps correspondent aux fermetures de marché (week-ends, nuits) — comportement attendu
- **Prix WTI** : Amplitude de prix observable sur la période initiale
- **Volume** : Distribution hétérogène selon les heures — pics en heures de trading actives (UTC)
- **Rendements** : Distribution quasi-normale centrée sur 0 — cohérent avec un actif financier liquide
- **Volatilité** : Visualisation de la volatilité glissante 24h permet d'identifier les périodes de tension

### Prochaines étapes

1. **Transformation** (`src/transformation/`) : normalisation, feature engineering (spreads, indicateurs techniques)
2. **Ingestion GDELT** : enrichissement avec données géopolitiques
3. **Calcul du Stress Index** : combinaison des signaux prix + GDELT
4. **Indexation Elasticsearch** : alimentation du dashboard de monitoring